# Import Library & Data


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load data hasil cleaning Lab
df_lab = pd.read_csv('df_lab_selected.csv')

In [3]:
df_lab.info()

<class 'pandas.DataFrame'>
RangeIndex: 4160 entries, 0 to 4159
Columns: 114 entries, NO_REG to wbc
dtypes: float64(111), int64(1), str(2)
memory usage: 3.6 MB


In [4]:
df_lab


,NO_REG,TGL_INPUT,NM_PERIKSA,aa do2,albumin,amilum,amoeba,amphetamine,anti hcv,asam urat,...,thc,trigliserida,troponin i high sensitive,tsh,tubex tf,ureum,urine protein,urobilinogen,warna,wbc
0,10579978,2026-02-05 09:14:00,urin lengkap,NaN,NaN,NaN,NaN,NaN,NaN,9.32,...,NaN,207.0,NaN,NaN,NaN,24.0,NaN,0.2,NaN,NaN
1,10580957,2026-02-06 09:06:00,lemak,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,205.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10581496,2026-02-06 19:25:00,ureum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,67.0,NaN,NaN,NaN,NaN
3,10582858,2026-02-03 11:56:00,lemak,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,206.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10583233,2026-02-11 18:19:00,ureum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,75.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4155,10646737,2026-06-30 16:41:00,hemoglobin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4156,10646760,2026-06-30 12:18:00,ureum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.015,NaN,NaN,23.0,NaN,NaN,NaN,6.7
4157,10646843,2026-06-30 13:53:00,hematologi rutin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.1
4158,10646876,2026-06-30 16:40:00,hemoglobin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Load data hasil cleaning Dokter
df_dokter = pd.read_csv('dokter_filtered.csv')


In [6]:
df_dokter.info()


<class 'pandas.DataFrame'>
RangeIndex: 8457 entries, 0 to 8456
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   NO_REG         8457 non-null   int64  
 1   TGL_LAHIR      8457 non-null   str    
 2   JENIS_KELAMIN  8457 non-null   str    
 3   UMUR_TAHUN     8457 non-null   float64
 4   UMUR_BULAN     8457 non-null   float64
 5   UMUR_HARI      8457 non-null   float64
 6   NM_UNIT        8457 non-null   str    
 7   SUBYEKTIF      8457 non-null   str    
 8   OBYEKTIF       8408 non-null   str    
 9   KD_PENYAKIT    8457 non-null   str    
 10  NM_PENYAKIT    8457 non-null   str    
 11  bulan          8457 non-null   str    
 12  TGL_INPUT      6910 non-null   str    
dtypes: float64(3), int64(1), str(9)
memory usage: 859.0 KB


# Merge Data


In [7]:
# Pastikan NO_REG berformat integer pada kedua tabel
df_lab['NO_REG'] = df_lab['NO_REG'].astype(int)
df_dokter['NO_REG'] = df_dokter['NO_REG'].astype(int)


In [8]:
# Buat indikator kehadiran data
df_lab['HAS_LAB'] = 1
df_dokter['HAS_DOCTOR'] = 1


C:\Users\ASUS\AppData\Local\Temp\ipykernel_31740\3826636030.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_lab['HAS_LAB'] = 1


In [9]:
# Outer merge pasien dokter dan pemeriksaan lab
df_merge = pd.merge(
    df_dokter,
    df_lab,
    on='NO_REG',
    how='outer'
)


In [10]:
df_merge['HAS_LAB'] = df_merge['HAS_LAB'].fillna(0)
df_merge['HAS_DOCTOR'] = df_merge['HAS_DOCTOR'].fillna(0)


In [11]:
df_merge.info()


<class 'pandas.DataFrame'>
RangeIndex: 11996 entries, 0 to 11995
Columns: 128 entries, NO_REG to HAS_LAB
dtypes: float64(116), int32(1), str(11)
memory usage: 11.7 MB


In [12]:
#df_merge.to_csv('df_merge_dokter_filter.csv', index=False)


In [13]:
df_merge[['HAS_LAB', 'HAS_DOCTOR']].value_counts()


HAS_LAB  HAS_DOCTOR
0.0      1.0           7836
1.0      0.0           3539
         1.0            621
Name: count, dtype: int64

# Feature Selection


In [14]:
# Seleksi hanya 13 parameter klinis + 1 kolom target
selected_features = [
    'NO_REG',
    'JENIS_KELAMIN',
    'UMUR_TAHUN',
    'cholesterol total',
    'creatinin',
    'glukosa darah puasa',
    'glukosa darah sewaktu',
    'hgb',
    'lymfosit%',
    'mch',
    'mchc',
    'mcv',
    'ureum',
    'wbc',
    'NM_UNIT'
]


In [15]:
df_clean = df_merge[selected_features].copy()


In [16]:
# Samakan nama kolom glukosa darah ke standar fitur (fbs & rbs)
df_clean = df_clean.rename(columns={
    'glukosa darah puasa': 'fbs',
    'glukosa darah sewaktu': 'rbs'
})


In [17]:
df_clean.info()


<class 'pandas.DataFrame'>
RangeIndex: 11996 entries, 0 to 11995
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   NO_REG             11996 non-null  int32  
 1   JENIS_KELAMIN      8457 non-null   str    
 2   UMUR_TAHUN         8457 non-null   float64
 3   cholesterol total  237 non-null    float64
 4   creatinin          1516 non-null   float64
 5   fbs                108 non-null    float64
 6   rbs                1436 non-null   float64
 7   hgb                3669 non-null   float64
 8   lymfosit%          3093 non-null   float64
 9   mch                3100 non-null   float64
 10  mchc               3100 non-null   float64
 11  mcv                3100 non-null   float64
 12  ureum              1340 non-null   float64
 13  wbc                3101 non-null   float64
 14  NM_UNIT            8457 non-null   str    
dtypes: float64(12), int32(1), str(2)
memory usage: 1.3 MB


# Data Transformation


In [18]:
# Encode gender (L=0, P=1, nan=2)
gender_encoder = LabelEncoder()
df_clean['JENIS_KELAMIN'] = gender_encoder.fit_transform(df_clean['JENIS_KELAMIN'].astype(str))


In [19]:
# Mapping label target spesialisasi
mapping_nm_unit = {
    'JANTUNG DAN PEMBULUH DARAH': 1,
    'SPES. PENYAKIT DALAM': 2,
    'SPES. PARU PARU': 3
}

df_clean['NM_UNIT'] = df_clean['NM_UNIT'].map(mapping_nm_unit)

# Pasien lab tanpa unit poli diisi 0 (Kelas 0: Kontrol / Sehat)
df_clean['NM_UNIT'] = df_clean['NM_UNIT'].fillna(0)


In [20]:
# Drop identifier NO_REG sebelum disimpan
df_clean = df_clean.drop(columns=['NO_REG'])


In [21]:
df_clean.info()


<class 'pandas.DataFrame'>
RangeIndex: 11996 entries, 0 to 11995
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   JENIS_KELAMIN      11996 non-null  int32  
 1   UMUR_TAHUN         8457 non-null   float64
 2   cholesterol total  237 non-null    float64
 3   creatinin          1516 non-null   float64
 4   fbs                108 non-null    float64
 5   rbs                1436 non-null   float64
 6   hgb                3669 non-null   float64
 7   lymfosit%          3093 non-null   float64
 8   mch                3100 non-null   float64
 9   mchc               3100 non-null   float64
 10  mcv                3100 non-null   float64
 11  ureum              1340 non-null   float64
 12  wbc                3101 non-null   float64
 13  NM_UNIT            11996 non-null  float64
dtypes: float64(13), int32(1)
memory usage: 1.2 MB


In [22]:
# Simpan dataset bersih akhir
df_clean.to_csv('merge_DocLab_Clean.csv', index=False)


In [23]:
# 1. Filter pasien yang memiliki Lab DAN Dokter
df_both = df_merge[(df_merge['HAS_LAB'] == 1) & (df_merge['HAS_DOCTOR'] == 1)]

# 2. Persebaran jumlah pasien per unit poli
print("=== JUMLAH PASIEN (LAB + DOKTER) ===")
print(df_both['NM_UNIT'].value_counts())

# 3. Persebaran dalam persentase
print("\n=== PERSENTASE (%) ===")
print((df_both['NM_UNIT'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')


=== JUMLAH PASIEN (LAB + DOKTER) ===
NM_UNIT
SPES. PENYAKIT DALAM          559
JANTUNG DAN PEMBULUH DARAH     52
SPES. PARU PARU                10
Name: count, dtype: int64

=== PERSENTASE (%) ===
NM_UNIT
SPES. PENYAKIT DALAM          90.02%
JANTUNG DAN PEMBULUH DARAH     8.37%
SPES. PARU PARU                1.61%
Name: proportion, dtype: str


In [24]:
summary_both = pd.DataFrame({
    'Jumlah Pasien': df_both['NM_UNIT'].value_counts(),
    'Persentase (%)': (df_both['NM_UNIT'].value_counts(normalize=True) * 100).round(2)
})
summary_both


,Jumlah Pasien,Persentase (%)
NM_UNIT,,
SPES. PENYAKIT DALAM,559,90.02
JANTUNG DAN PEMBULUH DARAH,52,8.37
SPES. PARU PARU,10,1.61


In [25]:
# Crosstab perbandingan status Lab per unit poli
tabel_persebaran = pd.crosstab(
    df_merge['NM_UNIT'].fillna('HANYA LAB (TANPA POLI / KELAS 0)'),
    df_merge['HAS_LAB'],
    margins=True,
    margins_name='TOTAL'
)

# Rename kolom agar mudah dibaca
tabel_persebaran.columns = ['Tanpa Lab (0)', 'Ada Lab (1)', 'Total Pasien']

# Tambah rasio ketersediaan lab (%)
tabel_persebaran['Rasio Lab (%)'] = (
    (tabel_persebaran['Ada Lab (1)'] / tabel_persebaran['Total Pasien']) * 100
).round(2).astype(str) + '%'

tabel_persebaran


,Tanpa Lab (0),Ada Lab (1),Total Pasien,Rasio Lab (%)
NM_UNIT,,,,
HANYA LAB (TANPA POLI / KELAS 0),0,3539,3539,100.0%
JANTUNG DAN PEMBULUH DARAH,1676,52,1728,3.01%
SPES. PARU PARU,1278,10,1288,0.78%
SPES. PENYAKIT DALAM,4882,559,5441,10.27%
TOTAL,7836,4160,11996,34.68%


In [26]:
# OPSI A: Ekstrak seluruh pasien yang punya data lab (4.160 baris, Kelas 0, 1, 2, 3)
df_with_lab = df_merge[df_merge['HAS_LAB'] == 1][selected_features].copy()
df_with_lab = df_with_lab.rename(columns={'glukosa darah puasa': 'fbs', 'glukosa darah sewaktu': 'rbs'})
df_with_lab['JENIS_KELAMIN'] = gender_encoder.fit_transform(df_with_lab['JENIS_KELAMIN'].astype(str))
df_with_lab['NM_UNIT'] = df_with_lab['NM_UNIT'].map(mapping_nm_unit).fillna(0)
df_with_lab = df_with_lab.drop(columns=['NO_REG'])
df_with_lab.to_csv('merge_DocLab_WithLab.csv', index=False)

# Data Manipulation

## Missing Value Handling

In [30]:
df_lab_data = pd.read_csv('merge_DocLab_WithLab.csv')
feature_cols = [c for c in df_lab_data.columns if c != 'NM_UNIT']

def impute_missing_by_class(group, features, k=5):
    X = group[features].copy()
    col_medians = X.median(numeric_only=True)
    fallback = col_medians.fillna(X.mean(numeric_only=True)).fillna(0)
    
    if len(X) <= 1:
        return X.fillna(fallback)
    
    X_reference = X.fillna(fallback)
    similarity = cosine_similarity(X_reference.values)
    np.fill_diagonal(similarity, -np.inf)
    
    imputed = X.copy()
    for col in features:
        missing_mask = imputed[col].isna()
        if not missing_mask.any():
            continue
            
        known_indices = np.where(~imputed[col].isna())[0]
        if len(known_indices) == 0:
            imputed[col] = imputed[col].fillna(fallback[col])
            continue
            
        for idx in np.where(missing_mask)[0]:
            sim_scores = similarity[idx, known_indices]
            top_k_indices = known_indices[np.argsort(sim_scores)[-min(k, len(sim_scores)):]]
            imputed.iloc[idx, imputed.columns.get_loc(col)] = group.iloc[top_k_indices][col].mean()
            
    return imputed.fillna(fallback)

processed_groups = []
for unit, group in df_lab_data.groupby('NM_UNIT', group_keys=False):
    imputed_subset = impute_missing_by_class(group, feature_cols)
    imputed_subset['NM_UNIT'] = unit
    processed_groups.append(imputed_subset)

df_imputed = pd.concat(processed_groups).sort_index()

# Pembulatan 1 angka di belakang koma (dan umur dibulatkan bulat)
df_imputed['UMUR_TAHUN'] = df_imputed['UMUR_TAHUN'].round(0)
numeric_cols = [c for c in df_imputed.columns if c not in ['JENIS_KELAMIN', 'UMUR_TAHUN', 'NM_UNIT']]
df_imputed[numeric_cols] = df_imputed[numeric_cols].round(1)

df_imputed.to_csv('merge_DocLab_WithLab_imputed.csv', index=False)
df_imputed.head()


,JENIS_KELAMIN,UMUR_TAHUN,cholesterol total,creatinin,fbs,rbs,hgb,lymfosit%,mch,mchc,mcv,ureum,wbc,NM_UNIT
0,0,38.0,193.7,1.1,101.0,138.0,12.4,15.6,28.8,32.0,511.0,24.0,11.4,1.0
1,1,60.0,221.9,1.1,134.0,188.0,11.7,29.0,29.6,32.8,90.3,45.4,7.3,2.0
2,0,79.0,178.2,2.2,145.9,194.0,9.8,24.6,27.5,32.3,85.0,67.0,7.5,2.0
3,1,67.0,208.7,1.2,160.1,154.0,9.8,25.3,27.7,32.2,85.7,45.6,7.3,2.0
4,0,68.0,183.7,2.1,110.2,477.0,10.3,15.4,28.3,33.3,85.0,75.0,17.1,2.0
